# Lab 03 · Limpieza y análisis exploratorio

*Análisis Avanzado de Datos con Python · Subsecretaría de Energía · Módulo 3*

Trabaja sobre tu propia copia del notebook. Todo lo que escribas queda en ella.

In [ ]:
#@title De qué se trata este lab { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d9edf7;border-left:6px solid #31708f;color:#0b3d62"><div style="font-weight:700;font-size:16px;margin-bottom:6px">De qué se trata este lab</div><strong>Preguntas que vamos a responder</strong>
<ul>
<li>Cómo se ve un archivo que llegó mal y qué hay que mirar antes de tocarlo</li>
<li>Qué hacer con las filas repetidas, los datos que faltan y los valores imposibles</li>
<li>Cuánto cambia un promedio según lo que uno decida hacer con los huecos</li>
<li>Qué tan bien quedó la limpieza, medido contra el dato original</li>
</ul>
<strong>Al terminar vas a poder</strong>
<ul>
<li>Diagnosticar una tabla sucia en cinco minutos, antes de calcular nada</li>
<li>Arreglar tipos, fechas con formato chileno y categorías escritas de varias formas</li>
<li>Detectar y tratar duplicados, nulos y valores extremos, dejando huella de lo que hiciste</li>
<li>Describir una variable con estadística que se entienda, y auditar tu propia limpieza</li>
</ul></div>"""))

In [ ]:
#@title Datos del curso { display-mode: "form" }
#@markdown Corre esta celda. Deja listos los archivos del Observatorio de Datos Energeticos.
import numpy as np, pandas as pd, os, json, sqlite3
if not os.path.exists("centrales.csv"):
    rng = np.random.default_rng(2026)
    centrales = pd.DataFrame([
     ("Central Rio Manso Alto","hidro","Biobio",420,2004),("Central Salto Verde","hidro","Los Lagos",310,1998),
     ("Central Aguas Claras","hidro","Biobio",180,2011),("Central Vega Azul","hidro","Los Lagos",95,2016),
     ("Central Tres Saltos","hidro","Biobio",260,1995),
     ("Parque Solar Pampa Alta","solar","Antofagasta",230,2019),("Parque Solar Llano Seco","solar","Atacama",180,2020),
     ("Parque Solar Sol Naciente","solar","Antofagasta",145,2021),("Parque Solar Quebrada Honda","solar","Atacama",95,2022),
     ("Parque Solar Altiplano","solar","Antofagasta",310,2023),
     ("Eolica Cerro Negro","eolica","Coquimbo",160,2017),("Eolica Punta Ventosa","eolica","Coquimbo",120,2018),
     ("Eolica Loma Fria","eolica","Valparaiso",85,2020),("Eolica Campo Abierto","eolica","Coquimbo",200,2021),
     ("Termoelectrica Bahia Norte","gas","Valparaiso",375,2008),("Termoelectrica Puerto Sur","gas","Biobio",290,2012),
     ("Termoelectrica Valle Central","gas","Metropolitana",210,2006),
     ("Carboelectrica Costa Brava","carbon","Biobio",480,2001),("Carboelectrica Roca Gris","carbon","Antofagasta",350,1999),
     ("Diesel Respaldo Cordillera","diesel","Metropolitana",45,2014),
    ], columns=["central","tecnologia","region","potencia_mw","anio_inicio"])
    fechas = pd.date_range("2024-01-01","2024-12-31",freq="D")
    perfil = np.array([0,0,0,0,0,0,.05,.18,.38,.58,.75,.87,.93,.9,.8,.63,.42,.2,.05,0,0,0,0,0])
    filas=[]
    for _,c in centrales.iterrows():
        p,t = c.potencia_mw, c.tecnologia
        for f in fechas:
            est = 1+0.25*np.cos(2*np.pi*(f.dayofyear-15)/365)
            if t=="solar": base = p*perfil*0.30*est*rng.uniform(.8,1.1)
            elif t=="eolica": base = p*0.36*rng.uniform(.15,1.6,24)
            elif t=="hidro": base = p*0.55*(2-est)*rng.uniform(.9,1.1,24)
            elif t=="gas": base = p*0.68*rng.uniform(.9,1.05,24)
            elif t=="carbon": base = p*0.65*rng.uniform(.95,1.02,24)
            else:
                base = np.zeros(24); base[18:23] = p*0.55*rng.uniform(.8,1,5)
            filas.append(pd.DataFrame({"fecha":f.strftime("%Y-%m-%d"),"hora":range(24),
                                       "central":c.central,"mwh":np.clip(base,0,p).round(2)}))
    centrales.to_csv("centrales.csv", index=False)
    pd.concat(filas, ignore_index=True).to_csv("generacion.csv", index=False)

    # Excel con dos hojas, la segunda con notas en texto libre
    with pd.ExcelWriter("centrales.xlsx") as w:
        centrales.to_excel(w, sheet_name="centrales", index=False)
        pd.DataFrame({"nota":["Potencias declaradas al 31 de diciembre de 2024",
                              "Las centrales de pasada se informan con su potencia maxima"]}
                     ).to_excel(w, sheet_name="notas", index=False)

    # Demanda por region, base de datos SQLite
    regs = ["Antofagasta","Atacama","Coquimbo","Valparaiso","Metropolitana","Biobio","Los Lagos"]
    pobl = [700000,320000,850000,1900000,8100000,1700000,900000]
    perfil_d = np.array([.72,.68,.66,.65,.66,.70,.78,.88,.95,.98,1.0,1.02,1.03,1.0,.97,.96,.97,1.0,1.06,1.10,1.08,.98,.88,.79])
    dem=[]
    for r,p in zip(regs,pobl):
        base_r = p/8000
        for f in fechas:
            inv = 1+0.18*np.cos(2*np.pi*(f.dayofyear-190)/365)
            finde = 0.92 if f.dayofweek>=5 else 1.0
            v = base_r*perfil_d*inv*finde*rng.uniform(.97,1.03,24)
            dem.append(pd.DataFrame({"fecha":f.strftime("%Y-%m-%d"),"hora":range(24),
                                     "region":r,"mwh":v.round(2)}))
    demanda = pd.concat(dem, ignore_index=True)
    demanda.to_csv("demanda.csv", index=False)
    con = sqlite3.connect("demanda.db")
    demanda.to_sql("demanda", con, index=False, if_exists="replace")
    pd.DataFrame({"region":regs,"poblacion":pobl}).to_sql("regiones", con, index=False, if_exists="replace")
    con.close()

    # Precios de nudo de enero, como los entregaria una API REST
    ene = demanda[demanda["fecha"].str.startswith("2024-01")]
    pr = ene.assign(precio_usd_mwh=(40 + ene["mwh"]/ene["mwh"].max()*110
                                    + rng.normal(0,6,len(ene))).clip(40,180).round(2))
    json.dump({"metadata":{"fuente":"Observatorio de Datos Energeticos",
                           "fecha_consulta":"2024-02-01","unidad":"USD por MWh"},
               "datos": pr[["fecha","hora","region","precio_usd_mwh"]].to_dict("records")},
              open("precios_nudo.json","w"))

# ---------------------------------------------------------------- demanda sucia
# La misma demanda, estropeada a proposito con siete defectos, que es el archivo
# con el que trabaja este lab. Se genera con semilla fija, asi que es siempre igual.
if not os.path.exists("demanda_sucia.csv"):
    base = pd.read_csv("demanda.csv")
    rng = np.random.default_rng(303)
    n, regs_n = len(base), base["region"].nunique()
    n_horas = n // regs_n

    # 1. bloques de horas sin dato, mas algunos sueltos, cerca del 3 por ciento
    nulos = set()
    while len(nulos) < int(n * 0.021):
        largo, reg = int(rng.integers(2, 13)), int(rng.integers(0, regs_n))
        ini = int(rng.integers(0, n_horas - largo))
        nulos.update((ini + k) * regs_n + reg for k in range(largo))
    while len(nulos) < int(n * 0.03):
        nulos.add(int(rng.integers(0, n)))
    idx_nulos = np.sort(np.fromiter(nulos, int))

    # 2. picos imposibles y valores negativos
    resto = rng.permutation(np.setdiff1d(np.arange(n), idx_nulos))
    picos, negativos = np.sort(resto[:40]), np.sort(resto[40:55])
    mwh = base["mwh"].to_numpy(float).copy()
    mwh[picos] = np.round(mwh[picos] * 10, 2)
    mwh[negativos] = np.round(-np.abs(mwh[negativos]) * rng.uniform(.1, 1, len(negativos)), 2)

    # 3. fecha en formato chileno y numero con coma decimal, los dos como texto
    fecha_txt = pd.to_datetime(base["fecha"]).dt.strftime("%d/%m/%Y")
    mwh_txt = np.array([f"{v:.2f}".replace(".", ",") for v in mwh], dtype=object)
    mwh_txt[idx_nulos] = ""

    # 4. la region escrita de varias formas
    VARIANTES = [str.upper, str.lower, lambda r: r + " ", lambda r: " " + r,
                 lambda r: r.lower() + " ", lambda r: "  " + r.upper()]
    region_txt = base["region"].to_numpy(dtype=object).copy()
    for pos in rng.choice(n, size=int(n * 0.01), replace=False):
        region_txt[pos] = VARIANTES[int(rng.integers(0, len(VARIANTES)))](region_txt[pos])

    suc = pd.DataFrame({"fecha": fecha_txt.to_numpy(dtype=object), "hora": base["hora"].to_numpy(),
                        "region": region_txt, "mwh": mwh_txt})

    # 5. veinte filas duplicadas exactas, pegadas a su original
    dup = np.sort(rng.choice(n, size=20, replace=False))
    orden = np.sort(np.concatenate([np.arange(n), dup]), kind="stable")
    # 6. separador punto y coma, como lo guarda Excel en Chile
    suc.iloc[orden].to_csv("demanda_sucia.csv", sep=";", index=False)
print("Datos listos, incluida la version sucia")

In [ ]:
import pandas as pd

# A proposito, sin ningun parametro que arregle nada. Solo el separador,
# porque sin el ni siquiera se ven las columnas.
sucia = pd.read_csv("demanda_sucia.csv", sep=";")

print(sucia.shape)
sucia.head()

## 1. Mirar antes de tocar

In [ ]:
# info muestra el tipo de cada columna y cuantos valores no nulos tiene.
sucia.info()

In [ ]:
# 2024 fue bisiesto, asi que son 366 dias por 24 horas por 7 regiones.
esperadas = 366 * 24 * 7

print("filas esperadas", esperadas)
print("filas del archivo", len(sucia))
print("sobran", len(sucia) - esperadas)

In [ ]:
# nunique cuenta valores distintos. Deberian ser 7 y son bastantes mas.
print("regiones distintas", sucia["region"].nunique())
print()
print(sucia["region"].value_counts().head(10))

In [ ]:
# Los valores vacios de mwh llegaron como nulos porque la columna es texto.
print("celdas vacias en mwh", sucia["mwh"].isna().sum())
print("tipo de la columna mwh", sucia["mwh"].dtype)
print("tipo de la columna fecha", sucia["fecha"].dtype)

In [ ]:
#@title Ojo con lo sucio que está este archivo { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#fdecea;border-left:6px solid #c0392b;color:#7b241c"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Ojo con lo sucio que está este archivo</div><p>Los siete defectos de este archivo son reales, pero venir todos juntos en el mismo CSV es un montaje del curso. En el trabajo aparecen de a uno y repartidos en meses.</p><p>Los tipos de defecto sí los vas a encontrar. El conjunto Generación Bruta Mensual SEN que publica la Comisión Nacional de Energía trae marca de orden de bytes al inicio y coma como separador decimal, o sea dos de los arreglos de este lab. Y la fecha chilena con el día primero está en cualquier planilla exportada acá.</p><p>Lo que no es realista es la densidad. Cuarenta y nueve formas de escribir siete regiones y un tres por ciento de celdas vacías es mucho más de lo que trae un archivo oficial.</p><p>Tampoco son reales las magnitudes. La demanda máxima del sistema de este lab es de 2.379,8 MWh por hora y la máxima real del Sistema Eléctrico Nacional en 2025 fue de 13.093,1 MWh por hora.</p></div>"""))

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Cuenta las variantes</strong>
<p>Muestra cuántas regiones distintas hay y cuáles no están escritas como corresponde. Pista, <code>value_counts()</code> las ordena de mayor a menor, y las siete correctas son las que pasan las ocho mil filas.</p></div>"""))

In [ ]:
# Tu turno
# Cambia el 5 por un numero mas grande para ver las variantes mal escritas.
print(sucia["region"].value_counts().head(5))

## 2. Tipos y conversiones

In [ ]:
# Sumar una columna de texto no da error, concatena, que es peor.
print(sucia["mwh"].head(3).sum()[:40])
print()
print("no es un numero, son tres textos pegados")

In [ ]:
# str da acceso a las operaciones de texto sobre toda la columna de una vez.
limpia = sucia.copy()
limpia["mwh"] = limpia["mwh"].str.replace(",", ".", regex=False).astype(float)

print(limpia["mwh"].dtype)
print("suma real", round(limpia["mwh"].sum(), 1), "MWh")

In [ ]:
# La forma corta, avisandole a pandas en la lectura en vez de arreglar despues.
prueba = pd.read_csv("demanda_sucia.csv", sep=";", decimal=",")

print("tipo de mwh leyendo con decimal", prueba["mwh"].dtype)

In [ ]:
# format le dice exactamente como esta escrita la fecha en el archivo.
limpia["fecha"] = pd.to_datetime(limpia["fecha"], format="%d/%m/%Y")

print(limpia["fecha"].dtype)
print("primer dia", limpia["fecha"].min().date(), " ultimo dia", limpia["fecha"].max().date())

In [ ]:
#@title Por que el formato de la fecha se escribe a mano { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#fdecea;border-left:6px solid #c0392b;color:#7b241c"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Por que el formato de la fecha se escribe a mano</div><p>Este archivo trae las fechas como 15/06/2024, que es el formato chileno. Si uno deja que pandas adivine, el 3 de junio lo puede leer como el 6 de marzo, sin avisar.</p><p>Con <code>format="%d/%m/%Y"</code> la lectura falla si alguna fila no calza, que es justo lo que uno quiere. Es preferible un error a un dato cambiado en silencio.</p></div>"""))

In [ ]:
# strip saca los espacios de los extremos y lower pasa todo a minusculas.
limpia["region"] = limpia["region"].str.strip().str.lower()

print("regiones distintas despues de normalizar", limpia["region"].nunique())
print(sorted(limpia["region"].unique()))

In [ ]:
# Y un diccionario devuelve cada una a su forma oficial.
CANONICAS = {"antofagasta": "Antofagasta", "atacama": "Atacama", "coquimbo": "Coquimbo",
             "valparaiso": "Valparaiso", "metropolitana": "Metropolitana",
             "biobio": "Biobio", "los lagos": "Los Lagos"}
limpia["region"] = limpia["region"].map(CANONICAS)

print("regiones", sorted(limpia["region"].unique()))
print("sin mapear", limpia["region"].isna().sum())

In [ ]:
# category guarda cada texto una sola vez y deja numeros en las filas.
antes = limpia["region"].memory_usage(deep=True) / 1024
limpia["region"] = limpia["region"].astype("category")
despues = limpia["region"].memory_usage(deep=True) / 1024

print(f"memoria de la columna region, antes {antes:.0f} KB, despues {despues:.0f} KB")

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Arregla en la lectura</strong>
<p>Vuelve a leer el archivo resolviendo dos conversiones en la propia lectura, el decimal con coma y la fecha con su formato. Pista, <code>read_csv</code> acepta <code>decimal</code>, <code>parse_dates</code> y <code>dayfirst</code>.</p></div>"""))

In [ ]:
# Tu turno
# Agrega los argumentos que faltan para que mwh llegue numerica y fecha como fecha.
otra = pd.read_csv("demanda_sucia.csv", sep=";")

print(otra.dtypes)

## 3. Filas repetidas

In [ ]:
# duplicated marca cada fila que ya aparecio antes, identica en todas las columnas.
print("filas duplicadas exactas", limpia.duplicated().sum())

In [ ]:
# keep=False marca las dos copias, no solo la segunda, para verlas en pareja.
repetidas = limpia[limpia.duplicated(keep=False)].sort_values(["fecha", "hora", "region"])

print(repetidas.shape[0], "filas involucradas")
repetidas.head(4)

In [ ]:
# drop_duplicates conserva la primera aparicion y descarta las repeticiones.
antes = len(limpia)
limpia = limpia.drop_duplicates()

print("antes", antes, " despues", len(limpia), " se fueron", antes - len(limpia))

In [ ]:
# subset limita la comparacion a las columnas que forman la clave logica.
CLAVE = ["fecha", "hora", "region"]

print("claves repetidas", limpia.duplicated(subset=CLAVE).sum())
print("filas", len(limpia), "y esperadas", 366 * 24 * 7)

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Comprueba la clave</strong>
<p>Verifica que no queden claves repetidas y cuenta cuántas filas hay por región. Las siete tienen que tener exactamente la misma cantidad.</p></div>"""))

In [ ]:
# Tu turno
# Cambia la columna por region y revisa que las siete tengan el mismo numero.
print(limpia.duplicated(subset=CLAVE).sum(), "claves repetidas")
print(limpia["hora"].value_counts().head(3))

## 4. Datos que faltan

In [ ]:
# isna marca los faltantes y sum los cuenta por columna.
print(limpia.isna().sum())
print()
print("porcentaje de mwh sin dato", round(100 * limpia["mwh"].isna().mean(), 2))

In [ ]:
# Para todo lo que viene, el orden importa.
limpia = limpia.sort_values(CLAVE).reset_index(drop=True)

# Los nulos no estan repartidos parejo, se concentran en bloques.
print(limpia.groupby("region", observed=True)["mwh"].apply(lambda s: s.isna().sum()))

In [ ]:
# Un dia concreto de Valparaiso, para ver como se ve un hueco de verdad.
hueco = limpia[(limpia["region"] == "Valparaiso")
               & (limpia["fecha"] == "2024-09-16")]

print(hueco[["hora", "mwh"]].to_string(index=False))

In [ ]:
# Estrategia 1, borrar las filas incompletas.
print("antes   ", len(limpia))
print("despues ", len(limpia.dropna(subset=["mwh"])))
print("se pierden", limpia["mwh"].isna().sum(), "filas y con ellas sus horas")

In [ ]:
# Estrategia 2, rellenar con un valor fijo, casi siempre una mala idea.
con_cero = limpia["mwh"].fillna(0)

print("promedio real     ", round(limpia["mwh"].mean(), 1))
print("promedio con ceros", round(con_cero.mean(), 1))

In [ ]:
# Estrategia 3, arrastrar el ultimo valor conocido, dentro de cada region.
adelante = limpia.groupby("region", observed=True)["mwh"].ffill()

# Estrategia 4, interpolar, o sea trazar una recta entre los dos extremos.
interpolada = limpia.groupby("region", observed=True)["mwh"].transform(
    lambda s: s.interpolate())

print("nulos que quedan, arrastrando ", adelante.isna().sum())
print("nulos que quedan, interpolando", interpolada.isna().sum())

In [ ]:
# Las cuatro estrategias sobre el mismo hueco, una al lado de la otra.
comparacion = pd.DataFrame({
    "hora": limpia.loc[hueco.index, "hora"],
    "original": limpia.loc[hueco.index, "mwh"],
    "con_cero": con_cero.loc[hueco.index],
    "arrastrado": adelante.loc[hueco.index],
    "interpolado": interpolada.loc[hueco.index],
}).round(1)

print(comparacion.head(14).to_string(index=False))

In [ ]:
# Guardamos primero que filas venian vacias, para poder auditarlas al final.
limpia["fue_nulo"] = limpia["mwh"].isna()
limpia["mwh"] = interpolada

print("filas marcadas como rellenadas", limpia["fue_nulo"].sum())
print("nulos que quedan", limpia["mwh"].isna().sum())

In [ ]:
#@title La decision que nadie ve en el informe { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#fdecea;border-left:6px solid #c0392b;color:#7b241c"><div style="font-weight:700;font-size:16px;margin-bottom:6px">La decision que nadie ve en el informe</div><p>Las cuatro estrategias dan cuatro promedios distintos sobre los mismos datos. Rellenar con cero baja el promedio, arrastrar lo aplana y interpolar lo suaviza.</p><p>Ninguna es la correcta siempre. Lo que no se puede hacer es elegir una sin dejar constancia, y por eso la columna <code>fue_nulo</code> se crea antes de rellenar. Permite responder después cuánto del resultado viene de datos medidos y cuánto de datos inventados por nosotros.</p></div>"""))

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Otra estrategia</strong>
<p>Rellena los nulos con el promedio de la misma región y esa misma hora del día. Suena más razonable que interpolar, porque respeta la forma del día. En el bloque 7 vamos a poder medir si de verdad lo es. Pista, agrupa por <code>region</code> y <code>hora</code> y usa <code>transform</code> con <code>mean</code>.</p></div>"""))

In [ ]:
# Tu turno
# Cambia la estrategia por el promedio de la region y la hora.
promedio_region = limpia.groupby("region", observed=True)["mwh"].transform("mean")

print(round(promedio_region.head(3), 1).tolist())

In [ ]:
#@title Hasta aca llega la primera clase { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d8f0df;border-left:6px solid #28a745;color:#14532d"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Hasta acá llega la primera clase</div><p>La tabla ya está usable. Tiene los tipos correctos, las regiones escritas de una sola forma, sin filas repetidas y sin huecos, y con la marca de qué filas fueron rellenadas.</p><p>La próxima clase parte mirando los valores que sobrevivieron y que igual son imposibles, y termina midiendo qué tan bien quedó esta limpieza contra el dato original.</p><p><strong>Cuando vuelvas, ejecuta el notebook desde arriba.</strong> Entorno de ejecución, Ejecutar todo. Colab no guarda el estado entre sesiones.</p></div>"""))

## 5. Valores imposibles

In [ ]:
# La regla de negocio mas simple que existe, y la mas efectiva.
negativos = limpia[limpia["mwh"] < 0]

print("filas con demanda negativa", len(negativos))
negativos[["fecha", "hora", "region", "mwh"]].head(3)

In [ ]:
# describe da la forma de la variable en ocho numeros.
limpia["mwh"].describe().round(1)

In [ ]:
# Metodo 1, rango intercuartilico, calculado dentro de cada region.
q1 = limpia.groupby("region", observed=True)["mwh"].transform(lambda s: s.quantile(0.25))
q3 = limpia.groupby("region", observed=True)["mwh"].transform(lambda s: s.quantile(0.75))
rango = q3 - q1
fuera_iqr = (limpia["mwh"] < q1 - 1.5 * rango) | (limpia["mwh"] > q3 + 1.5 * rango)

print("marcadas por rango intercuartilico", fuera_iqr.sum())

In [ ]:
# Metodo 2, z-score, que mide a cuantas desviaciones esta cada valor de su promedio.
media = limpia.groupby("region", observed=True)["mwh"].transform("mean")
desv = limpia.groupby("region", observed=True)["mwh"].transform("std")
z = (limpia["mwh"] - media) / desv
fuera_z = z.abs() > 3

print("marcadas por z-score", fuera_z.sum())

In [ ]:
# Metodo 3, Isolation Forest, que no mira una columna sino la combinacion.
from sklearn.ensemble import IsolationForest

X = limpia[["hora", "mwh"]]
modelo = IsolationForest(contamination=0.005, random_state=2026)
fuera_bosque = modelo.fit_predict(X) == -1

print("marcadas por Isolation Forest", fuera_bosque.sum())

In [ ]:
# Los tres metodos, lado a lado, y lo que mas importa, cuantos negativos pilla cada uno.
es_negativo = limpia["mwh"] < 0

for nombre, marca in [("rango intercuartilico", fuera_iqr), ("z-score", fuera_z),
                      ("Isolation Forest", fuera_bosque)]:
    print(f"{nombre:24} marca {marca.sum():5} filas, "
          f"y pilla {(marca & es_negativo).sum():2} de los {es_negativo.sum()} negativos")

In [ ]:
import numpy as np

# Convertimos a nulo lo que los dos metodos clasicos marcaron juntos, y los negativos.
sospechosas = (fuera_iqr & fuera_z) | es_negativo
limpia["mwh"] = np.where(sospechosas, np.nan, limpia["mwh"])

print("filas convertidas a nulo", sospechosas.sum())
print("nulos que quedan", limpia["mwh"].isna().sum())

In [ ]:
# Y se rellenan igual que antes, interpolando dentro de cada region.
limpia["mwh"] = limpia.groupby("region", observed=True)["mwh"].transform(
    lambda s: s.interpolate())

limpia["mwh"].describe().round(1)

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Mueve la sensibilidad</strong>
<p>Cambia <code>contamination</code> a 0.001, 0.005 y 0.01, y anota cuántas filas marca cada valor. Es la decisión más importante del método y no la toma el algoritmo, la tomas tú.</p></div>"""))

In [ ]:
# Tu turno
# Cambia el valor de contamination y vuelve a correr.
modelo = IsolationForest(contamination=0.005, random_state=2026)
marcadas = modelo.fit_predict(limpia[["hora", "mwh"]]) == -1

print("contamination 0.005 marca", marcadas.sum(), "filas")

## 6. Describir lo que quedo

In [ ]:
# describe acepta los percentiles que uno quiera, no solo los cuartiles.
limpia["mwh"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(1)

In [ ]:
# La asimetria dice hacia que lado se estira la cola de la distribucion.
print("asimetria", round(limpia["mwh"].skew(), 3))
print("mediana  ", round(limpia["mwh"].median(), 1))
print("promedio ", round(limpia["mwh"].mean(), 1))

In [ ]:
# describe tambien funciona despues de un groupby, una fila por grupo.
limpia.groupby("region", observed=True)["mwh"].describe().round(1)

In [ ]:
# Perfil horario promedio de cada region, una columna por region.
perfil = limpia.pivot_table(index="hora", columns="region", values="mwh",
                            aggfunc="mean", observed=True)

perfil.round(1).head(6)

In [ ]:
# Una franja horaria con pd.cut, como en el Modulo 2, y la demanda media de cada una.
limpia["franja"] = pd.cut(limpia["hora"], bins=[-1, 5, 11, 17, 23],
                          labels=["madrugada", "manana", "tarde", "noche"])

pd.crosstab(limpia["region"], limpia["franja"],
            values=limpia["mwh"], aggfunc="mean").round(1)

In [ ]:
#@title Un artefacto de los datos del curso { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#fdecea;border-left:6px solid #c0392b;color:#7b241c"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Un artefacto de los datos del curso</div><p>Si calculas la correlación entre las siete columnas del perfil horario, te va a dar 1,000 en todas las casillas. Eso no pasa en el sistema real, es un efecto de cómo se generaron estos datos, con el mismo perfil de día escalado por población.</p><p>En el Sistema Eléctrico Nacional las regiones no se parecen tanto. En el Norte Grande el consumo está dominado por la gran minería y la curva es casi plana, mientras en la zona centro predomina el consumo de clientes de distribución y la curva tiene mucha más forma.</p></div>"""))

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Los extremos por región</strong>
<p>Calcula el percentil 95 de <code>mwh</code> por región, que es la forma habitual de hablar de demanda alta sin quedarse con el máximo, que siempre es un caso raro. Pista, <code>quantile(0.95)</code> después de agrupar.</p></div>"""))

In [ ]:
# Tu turno
# Cambia mean por quantile(0.95) para ver la demanda alta de cada region.
print(limpia.groupby("region", observed=True)["mwh"].mean().round(1))

## 7. Contra la verdad

In [ ]:
# Este lab tiene una ventaja que la vida real no da, el dato original existe.
verdad = pd.read_csv("demanda.csv", parse_dates=["fecha"])

auditoria = pd.merge(limpia, verdad, on=CLAVE, how="inner", suffixes=("_limpia", "_real"))
print(auditoria.shape[0], "filas comparadas")

In [ ]:
# Error absoluto en MWh y error relativo en porcentaje, fila por fila.
auditoria["error"] = (auditoria["mwh_limpia"] - auditoria["mwh_real"]).abs()
auditoria["error_pct"] = 100 * auditoria["error"] / auditoria["mwh_real"]

print("error promedio   ", round(auditoria["error"].mean(), 2), "MWh")
print("error mediano    ", round(auditoria["error"].median(), 2), "MWh")
print("filas identicas  ", (auditoria["error"] < 0.01).sum(), "de", len(auditoria))

In [ ]:
# El error se concentra justo donde nosotros intervinimos.
print(auditoria.groupby("fue_nulo")["error"].agg(["count", "mean", "max"]).round(2))

In [ ]:
# Las diez filas donde mas nos equivocamos.
peores = auditoria.nlargest(10, "error")

peores[["fecha", "hora", "region", "mwh_real", "mwh_limpia", "error"]].head(5).round(1)

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Mide tu propia limpieza</strong>
<p>Compara el error promedio de las filas que rellenaste contra el de las que nunca tocaste, y sácale el porcentaje. Es el número que responde cuánto de tu tabla final es dato medido y cuánto es dato inventado por ti.</p></div>"""))

In [ ]:
# Tu turno
# Cambia error por error_pct y agrega el conteo, para ver el peso de cada grupo.
print(auditoria.groupby("fue_nulo")["error"].mean().round(2))

In [ ]:
#@title Lo que este bloque no se puede hacer en la vida real { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#fdecea;border-left:6px solid #c0392b;color:#7b241c"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Lo que este bloque no se puede hacer en la vida real</div><p>Acá pudimos medir la limpieza porque teníamos el archivo original. En el trabajo real eso no existe, el dato sucio es el único que hay.</p><p>Lo que sí se puede hacer siempre es lo de la columna <code>fue_nulo</code>, dejar registrado qué filas se tocaron y con qué criterio. Un informe que dice que el 3,0 por ciento de sus datos fue rellenado por interpolación es mucho más defendible que uno que no lo dice.</p></div>"""))

In [ ]:
#@title Puntos clave { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d8f0df;border-left:6px solid #28a745;color:#14532d"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Puntos clave</div><ul>
<li>Antes de tocar nada se mira la forma, los tipos y cuántos valores distintos hay donde debería haber pocos</li>
<li>Los problemas de tipo se arreglan mejor en la lectura que después, con <code>sep</code>, <code>decimal</code> y el formato de la fecha</li>
<li>Los duplicados se miran de dos formas, filas idénticas y claves repetidas</li>
<li>Cada estrategia de relleno cambia el resultado, así que la decisión se deja escrita y se marca qué filas se tocaron</li>
<li>Un valor extremo no es lo mismo que un valor imposible, y la regla de negocio pilla cosas que ningún método estadístico pilla</li>
<li>La limpieza se puede auditar, y el error se concentra donde uno intervino</li>
</ul>
<p>En el Lab 04 vamos a buscar patrones en la bitácora de mantenimiento, que es texto y fechas en vez de números.</p></div>"""))